In [56]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

# Load data
idp = pd.read_csv('/Users/jackzipper/QSS20/final_project_data/idp_dat_eastern_drc.csv')
iati = pd.read_csv('/Users/jackzipper/QSS20/final_project_data/iati-drc-cleaned.csv')

# Standardize province names
name_map = {'Nord-Kivu': 'Nord-kivu', 'Sud-Kivu': 'Sud-kivu', 'Ituri': 'Ituri'}
iati['location_name'] = iati['location_name'].str.strip().replace(name_map)

# Filter IATI to eastern provinces
eastern_provinces = ['Nord-kivu', 'Sud-kivu', 'Ituri']
iati_east = iati[iati['location_name'].isin(eastern_provinces)].copy()

# Convert dates
iati_east['day_start'] = pd.to_datetime(iati_east['day_start'], errors='coerce')
iati_east['day_end'] = pd.to_datetime(iati_east['day_end'], errors='coerce')
iati_east['day_length'] = (iati_east['day_end'] - iati_east['day_start']).dt.days

# Calculate monthly spend rate = total spend / project length in months
# Avoid division by zero
iati_east['monthly_spend'] = np.where(
    iati_east['day_length'] > 0,
    iati_east['spend'] / (iati_east['day_length'] / 30.44),
    0
)

idp['snapshot_month'] = pd.to_datetime(idp['snapshot_month'])

print(f"IATI eastern projects: {len(iati_east)}")
print(f"IDP panel rows: {len(idp)}")

# For each province-month in IDP data
records = []

for _, idp_row in idp.iterrows():
    province = idp_row['admin1_label']
    month = idp_row['snapshot_month']
    window_start = month - pd.DateOffset(months=6)

    province_projects = iati_east[iati_east['location_name'] == province]

    # New projects started in the 6 months prior to this month
    new_projects = province_projects[
        (province_projects['day_start'] >= window_start) &
        (province_projects['day_start'] < month)
    ]

    # All projects active in this month (control)
    active_projects = province_projects[
        (province_projects['day_start'] <= month) &
        (province_projects['day_end'] >= month)
    ]

    records.append({
        'admin1_label': province,
        'snapshot_month': month,
        'new_project_monthly_spend': new_projects['monthly_spend'].sum(),
        'new_project_count': len(new_projects),
        'total_active_monthly_spend': active_projects['monthly_spend'].sum(),
        'total_active_projects': len(active_projects)
    })

df_aid_panel = pd.DataFrame(records)

# Merge with IDP panel
df_merged = pd.merge(
    idp,
    df_aid_panel,
    on=['admin1_label', 'snapshot_month'],
    how='left'
)

# Log transformation: Aid spending is highly right-skewed since most of the aid dollars are concentrated in a few very large projects while most projects are small. 
df_merged['log_new_project_spend'] = np.log1p(df_merged['new_project_monthly_spend'])
df_merged['log_total_active_spend'] = np.log1p(df_merged['total_active_monthly_spend'])

df_merged.head()


IATI eastern projects: 1068
IDP panel rows: 57


,admin1_label,snapshot_month,total_displaced,num_sites_displaced,total_returnees,num_sites_returnees,net_monthly_flow,new_project_monthly_spend,new_project_count,total_active_monthly_spend,total_active_projects,log_new_project_spend,log_total_active_spend
0,Ituri,2021-09-01,98078.0,606.0,49189.0,311.0,48889.0,1.932914e+05,3,2.692629e+07,52,12.171960,17.108614
1,Ituri,2022-03-01,1951.0,11.0,0.0,0.0,1951.0,1.107629e+07,28,2.717668e+07,61,16.220318,17.117870
2,Ituri,2022-04-01,35136.0,2.0,0.0,0.0,35136.0,1.127534e+07,24,2.819298e+07,64,16.238129,17.154584
3,Ituri,2022-08-01,11362.0,1.0,42465.0,1.0,-31103.0,7.478776e+06,14,3.410636e+07,66,15.827580,17.344994
4,Ituri,2022-12-01,0.0,0.0,13200.0,1.0,-13200.0,4.584700e+06,18,3.347379e+07,61,15.338235,17.326273


In [57]:
# Create multi-index 
df_merged = df_merged.set_index(['admin1_label', 'snapshot_month'])

# Regression model: Entity FE only, robust standard errors
mod = PanelOLS(
    dependent=df_merged['net_monthly_flow'],
    exog=sm.add_constant(df_merged[['log_new_project_spend', 'log_total_active_spend']]),
    entity_effects=True,
    time_effects=False
)

res = mod.fit(cov_type='robust')
print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       net_monthly_flow   R-squared:                        0.0697
Estimator:                   PanelOLS   R-squared (Between):              0.4173
No. Observations:                  57   R-squared (Within):               0.0697
Date:                Tue, May 26 2026   R-squared (Overall):              0.0918
Time:                        11:57:24   Log-likelihood                   -708.89
Cov. Estimator:                Robust                                           
                                        F-statistic:                      1.9489
Entities:                           3   P-value                           0.1527
Avg Obs:                       19.000   Distribution:                    F(2,52)
Min Obs:                       15.000                                           
Max Obs:                       24.000   F-statistic (robust):             1.5798
                            

In [60]:
# Save regression summary to a text file
with open('/Users/jackzipper/QSS20/final_project_data/regression_results.txt', 'w') as f:
    f.write(str(res.summary))